In [1]:
import pandas as pd
from statsmodels.stats.proportion import proportion_confint
from pathlib import Path

BASE_DIR = Path(r"C:\Users\mrmik\Downloads\titanic")
df = pd.read_csv(BASE_DIR / "train.csv")

def get_ci(subset, label):
    count = subset['Survived'].sum()
    nobs = len(subset)
    # Использует интервал Уилсона
    lower, upper = proportion_confint(count, nobs, alpha=0.05, method='wilson')
    return {'Group': label, 'Successes': count, 'Total': nobs, 
            'Mean': count/nobs, 'CI_Lower': lower, 'CI_Upper': upper}

# Оцениваем разные подгруппы
results = []
results.append(get_ci(df, 'Все пассажиры'))
results.append(get_ci(df[df['Sex'] == 'female'], 'Женщины'))
results.append(get_ci(df[df['Pclass'] == 1], '1-й класс'))

ci_df = pd.DataFrame(results)
# Вывод
ci_df['Mean'] = ci_df['Mean'].apply("{:.2%}".format)
ci_df['CI_Lower'] = ci_df['CI_Lower'].apply("{:.2%}".format)
ci_df['CI_Upper'] = ci_df['CI_Upper'].apply("{:.2%}".format)
display(ci_df)

,Group,Successes,Total,Mean,CI_Lower,CI_Upper
0,Все пассажиры,342,891,38.38%,35.25%,41.62%
1,Женщины,233,314,74.20%,69.09%,78.73%
2,1-й класс,136,216,62.96%,56.35%,69.12%


In [2]:
from statsmodels.stats.proportion import proportions_ztest

# Тест 1: Женщины выживают чаще мужчин
surv_female = df[df['Sex'] == 'female']['Survived'].sum()
n_female = len(df[df['Sex'] == 'female'])
surv_male = df[df['Sex'] == 'male']['Survived'].sum()
n_male = len(df[df['Sex'] == 'male'])

stat, pval = proportions_ztest(
    count=[surv_female, surv_male], 
    nobs=[n_female, n_male], 
    alternative='larger' 
)

print(f"Тест 1 (женщины > мужчины): p-value = {pval:.2e}")
if pval < 0.05:
    print("Отвергаем H0. Женщины выживают статистически значимо чаще (p < 0.05).\n")
else:
    print("Не отвергаем H0.\n")


# Тест 2: Дети выживают чаще, чем взрослые
df_age = df.dropna(subset=['Age']).copy()
df_age['IsChild'] = df_age['Age'] < 18

surv_child = df_age[df_age['IsChild']]['Survived'].sum()
n_child = len(df_age[df_age['IsChild']])
surv_adult = df_age[~df_age['IsChild']]['Survived'].sum()
n_adult = len(df_age[~df_age['IsChild']])

stat2, pval2 = proportions_ztest(
    count=[surv_child, surv_adult], 
    nobs=[n_child, n_adult], 
    alternative='larger'
)

print(f"Тест 2 (дети > взрослые): p-value = {pval2:.4f}")
if pval2 < 0.05:
    print("Отвергаем H0. Дети выживают статистически значимо чаще (p < 0.05).")
else:
    print("Не отвергаем H0. Разница статистически незначима.")

Тест 1 (женщины > мужчины): p-value = 1.86e-59
Отвергаем H0. Женщины выживают статистически значимо чаще (p < 0.05).

Тест 2 (дети > взрослые): p-value = 0.0008
Отвергаем H0. Дети выживают статистически значимо чаще (p < 0.05).


In [3]:
import numpy as np
from scipy.stats import binom

np.random.seed(42)
n_participants = 1000
n_cards = 10
alpha = 0.01

# Моделируем количество правильных ответов для каждого (Биномиальное распределение)
# Так как экстрасенсов нет, p=0.5 для всех
correct_guesses = np.random.binomial(n=n_cards, p=0.5, size=n_participants)

# Считаем p-value для каждого участника.
# p-value здесь - это вероятность угадать ТАК ЖЕ или ЛУЧШЕ при условии, что он не экстрасенс.
# Используем функцию выживания (sf = 1 - cdf), так как нас интересует правый хвост (много угадываний).
p_values = binom.sf(correct_guesses - 1, n_cards, 0.5)

# 1. Без множественной проверки
# Экстрасенсы - это те, у кого p-value < 0.01
psychics_no_correction = (p_values < alpha).sum()

# 2. С поправкой Бонферрони (правило FWER)
# Делим уровень значимости на количество гипотез (участников)
alpha_bonferroni = alpha / n_participants
psychics_bonferroni = (p_values < alpha_bonferroni).sum()

print(f"Участников, которых признали экстрасенсами БЕЗ поправки (p < 0.01): {psychics_no_correction}")
print(f"Участников, которых признали экстрасенсами С поправкой Бонферрони (p < {alpha_bonferroni}): {psychics_bonferroni}")

Участников, которых признали экстрасенсами БЕЗ поправки (p < 0.01): 1
Участников, которых признали экстрасенсами С поправкой Бонферрони (p < 1e-05): 0
